# 06. Retrieval and Tool Versatility

This notebook shows how a single agent can use tools of very different kinds together.
Tools are ordinary Python functions — they can read local files, call external APIs,
run computations, or anything else Python can do. The agent decides which tool to call
and when, based on the task at hand.

## Why tool versatility matters in DH
A real archival pipeline rarely stays inside one system. You extract a place name from a letter,
look it up in a gazetteer, find biographical context for a person on Wikipedia, and then check
whether the source document is digitised in a public archive. Each of those steps is a tool.
The agent's value is in coordinating them.

## Three tool categories demonstrated

| Tool | Category | Auth required? |
|------|----------|----------------|
| `search_documents` | Local file retrieval (RAG) | None |
| `wikipedia_summary` | REST API — entity lookup | None |
| `geocode_place` | REST API — spatial enrichment | None (User-Agent header only) |

All three are implemented with Python stdlib — no extra packages needed.

## Concepts
- Retrieval-Augmented Generation (RAG) with local files
- API tools with no authentication
- Responsible API use (rate limits, User-Agent headers)
- Multi-tool agents
- Grounding answers in both local evidence and external knowledge

## Dataset
The three sample letters in `data/` are used as the local document collection.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip`
from the repository root to the current working directory using the Files panel,
then rerun the setup cell below.

## Further reading
- Tools: https://openai.github.io/openai-agents-python/tools/
- Agents: https://openai.github.io/openai-agents-python/agents
- Wikipedia REST API: https://en.wikipedia.org/api/rest_v1/
- Nominatim usage policy: https://operations.osmfoundation.org/policies/nominatim/

In [1]:
import json
import os
import sys
import time
import urllib.parse
import urllib.request
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, set_tracing_export_api_key, trace

DEFAULT_MODEL = 'gpt-5.4-mini'

In [2]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found.')

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])

DOCUMENTS: list[dict[str, str]] = []
for path in sorted(data_dir.glob('*.txt')):
    DOCUMENTS.append({
        'filename': path.name,
        'text': path.read_text(encoding='utf-8').strip(),
    })

print(f'Loaded {len(DOCUMENTS)} documents:')
for doc in DOCUMENTS:
    print(' -', doc['filename'])

Loaded 6 documents:
 - letter_01_madrid_1871.txt
 - letter_02_seville_1894.txt
 - letter_03_ocr_fragment.txt
 - letter_04_barcelona_1902.txt
 - letter_05_porto_1888.txt
 - letter_06_ocr_table.txt


## Tool 1: Local document search (RAG)

Retrieval-Augmented Generation (RAG) means giving the agent a way to look up relevant
passages from a document collection rather than relying on the model's training data.

This is the simplest possible RAG implementation: keyword search over lines.
In production you would replace line-level substring matching with embedding-based
semantic search, but the agent interface stays identical — a function that takes a
query string and returns matching text.

The agent does not have the document content in its context window.
It must explicitly call the tool to retrieve relevant passages — that is what makes it retrieval-augmented.

In [3]:
@function_tool
def search_documents(query: str) -> str:
    """Search the local document archive for lines containing the query term.
    Returns matching lines with their source filename.
    """
    hits = []
    for doc in DOCUMENTS:
        for line in doc['text'].splitlines():
            if query.lower() in line.lower() and line.strip():
                hits.append(f"[{doc['filename']}] {line.strip()}")
    if not hits:
        return f"No documents matched '{query}'."
    return '\n'.join(hits)

# Quick test before wiring it to an agent.
print(search_documents.on_invoke_tool)
print()
print('--- test: query "Madrid" ---')
# Call the underlying function directly for testing (not through the agent).
for doc in DOCUMENTS:
    for line in doc['text'].splitlines():
        if 'madrid' in line.lower() and line.strip():
            print(f"  [{doc['filename']}] {line.strip()}")


--- test: query "Madrid" ---
  [letter_01_madrid_1871.txt] Madrid, 4 April 1871
  [letter_01_madrid_1871.txt] In 1871, Maria Gomez wrote from Madrid to her brother in Valencia about the exhibition and the costs of travel. She also said she hoped the spring weather would make the journey easier when she next visited.


## Tool 2: Wikipedia entity lookup (REST API, no key)

The [Wikipedia REST API](https://en.wikipedia.org/api/rest_v1/) provides plain-text
summaries for any article — no API key, no sign-up, one HTTP GET request.

For DH workflows this is a lightweight enrichment step: you extract a person or place
name from a historical document and immediately fetch biographical or geographic context.

**Error handling note:** The tool catches HTTP errors gracefully and returns a descriptive
message so the agent can continue even when a lookup fails.

In [4]:
def fetch_wikipedia_summary(topic: str) -> str:
    encoded = urllib.parse.quote(topic.replace(' ', '_'))
    url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{encoded}'
    req = urllib.request.Request(url, headers={'User-Agent': 'agentic-example/0.1'})
    try:
        with urllib.request.urlopen(req, timeout=8) as response:
            data = json.loads(response.read())
            extract = data.get('extract', '')
            return extract if extract else f"No Wikipedia summary found for '{topic}'."
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            return f"No Wikipedia article found for '{topic}'."
        return f"Wikipedia lookup failed ({exc.code}): {exc.reason}"
    except Exception as exc:
        return f"Wikipedia lookup failed: {exc}"

@function_tool
def wikipedia_summary(topic: str) -> str:
    """Fetch a short Wikipedia summary for a person, place, or concept.
    Returns the introductory paragraph from the Wikipedia article.
    """
    return fetch_wikipedia_summary(topic)

# Quick test.
print(wikipedia_summary.name)

# Live test via plain Python helper (not through the agent wrapper):
print(fetch_wikipedia_summary('Madrid')[:300])

wikipedia_summary
Madrid is the capital and most populous city of Spain. It had a population of over 3.4 million in the city proper in 2025, and a metropolitan area population of approximately 6.8 million. Madrid is the second-largest city in the European Union (EU), after Berlin, and its metropolitan area is the sec


## Tool 3: Nominatim geocoding (REST API, no key)

[Nominatim](https://nominatim.org/) is the OpenStreetMap geocoding service. It turns
a place name into geographic coordinates — no API key required, only a `User-Agent` header.

**Why this matters for DH:** Geocoding place names from historical documents is the first
step toward spatial humanities — mapping correspondence networks, trade routes, migration
patterns, and more.

**Responsible API use:** Nominatim's usage policy asks for a maximum of 1 request per second
and a meaningful `User-Agent` string. The tool includes a short sleep to respect this limit.
Respecting rate limits and identifying your client are standard expectations when using any
public API.

In [5]:
def fetch_geocode(place_name: str) -> str:
    time.sleep(1)  # Respect Nominatim's 1 req/sec policy.
    params = urllib.parse.urlencode({'q': place_name, 'format': 'json', 'limit': '1'})
    url = f'https://nominatim.openstreetmap.org/search?{params}'
    req = urllib.request.Request(url, headers={'User-Agent': 'agentic-example/0.1'})
    try:
        with urllib.request.urlopen(req, timeout=8) as response:
            results = json.loads(response.read())
        if not results:
            return f"No coordinates found for '{place_name}'."
        r = results[0]
        return (
            f"{place_name}: lat={r['lat']}, lon={r['lon']}\n"
            f"Resolved as: {r.get('display_name', 'unknown')}"
        )
    except Exception as exc:
        return f"Geocoding failed for '{place_name}': {exc}"

@function_tool
def geocode_place(place_name: str) -> str:
    """Find the geographic coordinates (latitude, longitude) for a place name.
    Uses OpenStreetMap Nominatim. Returns coordinates and the full resolved address.
    """
    return fetch_geocode(place_name)

print(geocode_place.name)
print(fetch_geocode('Madrid'))

geocode_place
Madrid: lat=40.4167820, lon=-3.7035070
Resolved as: Madrid, Comunidad de Madrid, España


## Step 4: Build the multi-tool agent

All three tools are registered on a single agent. The agent's instructions tell it
what each tool is for so it can decide when to use which one.

Notice that the agent does not know in advance which tools it will need — the
decision emerges from the query. That is the core of agentic behaviour.

In [6]:
dh_research_agent = Agent(
    name='DH Research Assistant',
    instructions=(
        'You are a research assistant for digital humanities. '
        'You have access to three tools:\n'
        '1. search_documents — search the local archive of historical letters.\n'
        '2. wikipedia_summary — fetch a Wikipedia summary for a person or place.\n'
        '3. geocode_place — find geographic coordinates for a place name.\n\n'
        'Always ground your answers in evidence. '
        'If a tool returns no result, say so explicitly rather than guessing.'
    ),
    tools=[search_documents, wikipedia_summary, geocode_place],
    model=DEFAULT_MODEL,
)

dh_research_agent

Agent(name='DH Research Assistant', handoff_description=None, tools=[FunctionTool(name='search_documents', description='Search the local document archive for lines containing the query term.\nReturns matching lines with their source filename.', params_json_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_documents_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x799f53a38f50>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False), FunctionTool(name='wikipedia_summary', description='Fetch a short Wikipedia summary for a person, place, or concept.\nReturns the introductory paragraph from the Wikipedia article.', params_json_schema={'properties': {'topic': {'title': 'Topic'

## Demo 1: Local search

Ask a question whose answer lives entirely in the local documents.
The agent should use `search_documents` and cite the source file.

In [7]:
query1 = 'Who wrote a letter from Madrid in 1871, and what was it about?'

with trace('RAG demo 1 — local search'):
    result1 = await Runner.run(dh_research_agent, query1)

print(result1.final_output)

It was written by **Maria Gomez**.  

The letter, from **Madrid in 1871**, was about **an exhibition** and **the costs of travel**; she also mentioned hoping the **spring weather** would make a future trip easier.


## Demo 2: Mixed — local search + Wikipedia enrichment

Ask a question that requires finding a name in the local archive *and* looking it up externally.
Watch the trace to see the agent call two different tools in sequence.

In [8]:
query2 = (
    'Find all place names mentioned in the local letters. '
    'Then fetch a short Wikipedia summary for each place.'
)

with trace('RAG demo 2 — local search + Wikipedia'):
    result2 = await Runner.run(dh_research_agent, query2)

print(result2.final_output)

Place names found in the letters:
- Madrid
- Valencia
- Seville
- Barcelona
- Girona
- Porto
- Lisbon
- Braga
- Cádiz
- Toledo

Short Wikipedia summaries:
- Madrid — Capital and most populous city of Spain; political, economic, and cultural center.
- Valencia — Capital of the Valencian Community in Spain, on the Mediterranean coast.
- Seville — Capital and largest city of Andalusia in southwest Spain.
- Barcelona — Major city on Spain’s northeastern coast; capital of Catalonia.
- Girona — Capital of the Province of Girona in Catalonia, Spain; known for its preserved old quarter.
- Porto — Second-largest city in Portugal, known in English as Oporto.
- Lisbon — Capital and most populous city of Portugal, on the River Tagus.
- Braga — City in northwestern Portugal; capital of the Braga district.
- Cádiz — Historic port city in Andalusia, Spain, on the Atlantic Ocean.
- Toledo — Ambiguous in Wikipedia; commonly refers to Toledo, Spain, or Toledo, Ohio.


## Demo 3: Full pipeline — search + Wikipedia + geocoding

Ask a question that uses all three tools. The agent searches the archive for places,
looks each one up on Wikipedia for context, and geocodes them for spatial analysis.

This is a realistic starting point for building a correspondence map in a DH project.

In [9]:
query3 = (
    'Search the local letters for place names. '
    'For each unique place: get its Wikipedia summary and its coordinates. '
    'Present the results as a structured list.'
)

with trace('RAG demo 3 — all three tools'):
    result3 = await Runner.run(dh_research_agent, query3)

print(result3.final_output)

Found these unique place names in the letters: Madrid, Valencia, Barcelona, Girona.

1. Madrid
   - Evidence: “In 1871, Maria Gomez wrote from Madrid…” (`letter_01_madrid_1871.txt`)
   - Wikipedia: capital and most populous city of Spain; political, economic, and cultural center.
   - Coordinates: 40.4167820, -3.7035070

2. Valencia
   - Evidence: “...wrote from Madrid to her brother in Valencia...” (`letter_01_madrid_1871.txt`)
   - Wikipedia: capital of the Valencian Community and province of the same name in Spain, on the Mediterranean coast.
   - Coordinates: 39.4697065, -0.3763353

3. Barcelona
   - Evidence: “...I visited the reading room on Tuesday...” (`letter_04_barcelona_1902.txt`) and the filename context.
   - Wikipedia: city on the northeastern coast of Spain; capital and largest city of Catalonia.
   - Coordinates: 41.3825802, 2.1770730

4. Girona
   - Evidence: “...addressed to the publisher Eduard Domènech in Girona.” (`letter_04_barcelona_1902.txt`)
   - Wikipedia: cap

## Demo 4: Graceful no-result handling

Ask about something that is not in the local archive. A well-behaved agent should
acknowledge the absence of evidence rather than hallucinating an answer.

In [10]:
query4 = 'Find any letters written from Buenos Aires in the 1880s.'

with trace('RAG demo 4 — no result'):
    result4 = await Runner.run(dh_research_agent, query4)

print(result4.final_output)

I couldn’t find any matching letters in the archive.

I searched for:
- “Buenos Aires 1880s”
- “Buenos Aires”
- “1880”

All returned no documents matched.


## Discussion: tool categories and when to use each

| Category | Example | When to use |
|----------|---------|-------------|
| **Local file search** | `search_documents` | Your collection is private, large, or not indexed externally. Keyword search is transparent and fast; replace with embedding search when semantic similarity matters more than exact terms. |
| **No-auth REST API** | Wikipedia, Nominatim | Quick enrichment of extracted entities. No sign-up required — immediately runnable and suitable for prototyping. |
| **Authenticated REST API** | OpenLibrary, OCLC WorldCat, Europeana | Pipelines where you need authoritative bibliographic or archival data. Same tool pattern, just add the API key to `.env`. |
| **Computation tool** | Date normaliser, string cleaner | Deterministic transformations you don't want the model to guess — wrap as a tool and the result is reproducible. |
| **Database tool** | SQLite, PostgreSQL | When your collection is structured and queryable. The tool executes a SQL query and returns rows; the agent decides what to query. |

### Keyword search vs. embedding-based retrieval

The `search_documents` tool here does line-level substring matching. This is:
- **Transparent**: you can read exactly what matched and why.
- **Fast**: no model calls, no vector index.
- **Brittle**: misses synonyms, paraphrases, and OCR variants.

Embedding-based retrieval (encoding documents and queries as vectors, then finding
nearest neighbours) handles semantic similarity but adds infrastructure and cost.
Keyword search is the right starting point for most DH projects.

### Other APIs worth exploring
- **Internet Archive** (`archive.org/advancedsearch.php`): check whether a source document is already digitised. No key required.
- **Europeana REST API**: search European cultural heritage collections. Free tier available.
- **Wikidata SPARQL**: structured linked data for historical entities (requires SPARQL knowledge — recommended as an advanced topic).

## Exercise

The `search_documents` tool returns individual lines. A single line often lacks context
— you want the surrounding paragraph.

Modify `search_documents` so that for each matching line it also returns the two lines
immediately before and after it (a context window of ±2 lines). Deduplicate overlapping
windows when multiple lines in the same paragraph match.

Test it by searching for `"exhibition"` and comparing the output before and after the change.

## Bonus exercise

Add a fourth tool that searches the **Internet Archive** full-text search
(`https://archive.org/advancedsearch.php?q={query}&output=json`) and returns the top
three matching item titles and their archive URLs. Attach it to `dh_research_agent`
and ask: *"Are there any digitised editions related to the Seville 1894 letter?"*